In [1]:
!pip install datasets pyarrow pandas

### Load the dataset

In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "ButterChicken98/plantvillage-image-text-pairs",
    split="train"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/365 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/344M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20638 [00:00<?, ? examples/s]

In [3]:
df = dataset.to_pandas()
df.head()

,image,caption,captions
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato Late blight,[A tomato leaf showing dark brown lesions and ...
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato mosaic virus,[A tomato leaf with mosaic-like patterns of li...
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Pepper bell healthy,"[A fresh green bell pepper leaf with a smooth,..."


In [4]:
df.rename(columns={'caption': 'class_name'}, inplace=True)

In [5]:
print(df.columns)

Index(['image', 'class_name', 'captions'], dtype='object')


### Class and class name

In [6]:
print("Total classes:", df['class_name'].nunique())
print(df['class_name'].unique())

Total classes: 15
['Tomato healthy' 'Tomato Late blight' 'Tomato mosaic virus'
 'Pepper bell healthy' 'Potato Early blight' 'Tomato Early blight'
 'Tomato YellowLeaf Curl Virus' 'Tomato Target Spot'
 'Pepper bell Bacterial spot' 'Tomato Septoria leaf spot'
 'Tomato Spider mites Two spotted spider mite' 'Tomato Bacterial spot'
 'Potato Late blight' 'Tomato Leaf Mold' 'Potato healthy']


In [7]:
df['class_name'].value_counts()

,count
class_name,
Tomato YellowLeaf Curl Virus,3208
Tomato Bacterial spot,2127
Tomato Late blight,1909
Tomato Septoria leaf spot,1771
Tomato Spider mites Two spotted spider mite,1676
Tomato healthy,1591
Pepper bell healthy,1478
Tomato Target Spot,1404
Potato Late blight,1000


In [8]:
df['num_captions'] = df['captions'].apply(len)
df['num_captions'].value_counts()

,count
num_captions,
4,20638


In [9]:
all_captions = df['captions'].explode()

print("Total symptom descriptions:", len(all_captions))
print("Unique descriptions:", all_captions.nunique())

Total symptom descriptions: 82552
Unique descriptions: 60


In [10]:
unique_desc_per_class = df.explode('captions') \
                          .groupby('class_name')['captions'] \
                          .nunique()

print(unique_desc_per_class)

class_name
Pepper bell Bacterial spot                     4
Pepper bell healthy                            4
Potato Early blight                            4
Potato Late blight                             4
Potato healthy                                 4
Tomato Bacterial spot                          4
Tomato Early blight                            4
Tomato Late blight                             4
Tomato Leaf Mold                               4
Tomato Septoria leaf spot                      4
Tomato Spider mites Two spotted spider mite    4
Tomato Target Spot                             4
Tomato YellowLeaf Curl Virus                   4
Tomato healthy                                 4
Tomato mosaic virus                            4
Name: captions, dtype: int64


In [11]:
from collections import defaultdict

desc_map = defaultdict(set)

for _, row in df.iterrows():
    for cap in row['captions']:
        desc_map[cap].add(row['class_name'])

duplicates = {k:v for k,v in desc_map.items() if len(v) > 1}

print("Descriptions shared across classes:", len(duplicates))

Descriptions shared across classes: 0
